# Topic 3.4 - 子查询

## 1. 子查询 - FROM 子句中的子查询

子查询是 SQL 中的一种语法结构，和 CTE 类似，子查询也是 SQL 里面套 SQL：

- 具体来说，子查询分为三种类型:

    - `FROM` 子句中的子查询
    - `SELECT` 子句中的子查询
    - `WHERE` 子句中的子查询
 - 我们先来讲一下 `FROM` 子句中的子查询，因为它和我们上一节讲的 CTE 的概念最为接近

`FROM` 子句中的子查询和 CTE 的概念非常相似，都是在一个 SQL 的查询结果上继续查询数据，它们的语法区别如下：

- CTE 的语法是：

```sql
WITH
临时表1 AS (
    SELECT ...
),
临时表2 AS (
    SELECT ...
),
...
临时表N AS (
    SELECT ...
)
SELECT ...
FROM 临时表1
JOIN 临时表2 ON ...
JOIN ... ON ...
```

- `FROM` 子句中的子查询的语法是：

```sql
SELECT ...
FROM (
    SELECT ...
) AS 子查询表1
JOIN (
    SELECT ...
) AS 子查询表2 ON 子查询表1.连接键 = 子查询表2.连接键
JOIN (
    SELECT ...
) AS 子查询表3 ON 子查询表1.连接键 = 子查询表3.连接键
...
```

子查询的特征是：

- 支持一个查询中定义多个子查询，每个子查询都是一个完整的 SQL 语句，可以包含各种子句，例如 `WHERE`、`GROUP BY`、`ORDER BY`、`JOIN`、`UNION` 等等，甚至可以嵌套子查询
- 但是，**子查询之间不可以相互引用**，也就是说，在定义子查询的时候，不能使用之前定义的子查询来查询数据，哪怕我们给子查询起了别名也不行
- 子查询只能通过 JOIN 的方式来连接在一起，不能使用 UNION 来连接在一起，否则会报错

我们来简单对比一下 CTE 和 `FROM` 子句中的子查询：

- CTE 不支持 CTE 嵌套，但是子查询是支持子查询嵌套的
- CTE 支持 CTE 之间相互引用，但是子查询不支持子查询之间相互引用
- CTE 支持 CTE 之间通过 JOIN 或者 UNION 来连接在一起，但是子查询只能通过 JOIN 来连接在一起，不能使用 UNION 来连接在一起

但是大家会发现，子查询远不如 CTE 灵活好用

- 一方面是，上面我们对比过 CTE 和子查询，CTE 的功能更强大，支持 CTE 之间相互引用，支持 CTE 之间通过 JOIN 或者 UNION 来连接在一起，而子查询不支持这些功能
- 另一方面的原因是，CTE 是先准备子表，再进行查询，而 `FROM` 子句中的子查询缺正好相反，得先想好自己要查询什么样的数据，才去写子查询，这个思维是有点反人类的
- 因此，如果没有专门要求，我们更推荐大家使用 CTE，来实现从一个查询结果上继续查询数据的功能



这里我们来看一个例子，使用 CTE 和 `FROM` 子句中的子查询，来实现同样的功能：

- 查询出消费金额大于所在国家平均消费金额的客户的 CustomerId、FirstName、LastName、Country、总消费金额，并按总消费金额降序排列：

- 使用 CTE 来实现：

In [2]:
%%sql
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        C.Country,
        I.Total AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
),
CountryAverageSpending AS (
    SELECT
        Country,
        AVG(TotalSpending) AS AverageSpending
    FROM CustomerSpending
    GROUP BY Country
)
SELECT
    CS.CustomerId,
    CS.FirstName,
    CS.LastName,
    CS.Country,
    CS.TotalSpending,
    CAS.AverageSpending
FROM CustomerSpending       AS CS
JOIN CountryAverageSpending AS CAS ON CS.Country = CAS.Country
WHERE CS.TotalSpending > CAS.AverageSpending
ORDER BY CS.TotalSpending DESC

,CustomerId,FirstName,LastName,Country,TotalSpending,AverageSpending
0,6,Helena,Holý,Czech Republic,25.86,6.445714
1,26,Richard,Cunningham,USA,23.86,5.747912
2,45,Ladislav,Kovács,Hungary,21.86,6.517142
3,46,Hugh,O'Reilly,Ireland,21.86,6.517142
4,7,Astrid,Gruber,Austria,18.86,6.088571
...,...,...,...,...,...,...
167,52,Emma,Jones,United Kingdom,5.94,5.374285
168,53,Phil,Hughes,United Kingdom,5.94,5.374285
169,54,Steve,Murray,United Kingdom,5.94,5.374285
170,55,Mark,Taylor,Australia,5.94,5.374285


- 使用 `FROM` 子句中的子查询来实现：

In [5]:
%%sql
SELECT
    CS.CustomerId,
    CS.FirstName,
    CS.LastName,
    CS.Country,
    CS.TotalSpending,
    CAS.AverageSpending
FROM (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        C.Country,
        I.Total AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
) AS CS
JOIN (
    SELECT
    Country,
    AVG(TotalSpending) AS AverageSpending
    FROM (
        SELECT
            C.CustomerId,
            C.FirstName,
            C.LastName,
            C.Country,
            I.Total AS TotalSpending
        FROM Customer AS C
        JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    ) AS CS2
    GROUP BY Country
) AS CAS ON CS.Country = CAS.Country
WHERE CS.TotalSpending > CAS.AverageSpending
ORDER BY CS.TotalSpending DESC

,CustomerId,FirstName,LastName,Country,TotalSpending,AverageSpending
0,6,Helena,Holý,Czech Republic,25.86,6.445714
1,26,Richard,Cunningham,USA,23.86,5.747912
2,45,Ladislav,Kovács,Hungary,21.86,6.517142
3,46,Hugh,O'Reilly,Ireland,21.86,6.517142
4,7,Astrid,Gruber,Austria,18.86,6.088571
...,...,...,...,...,...,...
167,52,Emma,Jones,United Kingdom,5.94,5.374285
168,53,Phil,Hughes,United Kingdom,5.94,5.374285
169,54,Steve,Murray,United Kingdom,5.94,5.374285
170,55,Mark,Taylor,Australia,5.94,5.374285


## 2. 子查询 - SELECT 子句中的子查询

我们之前介绍过，在 `SELECT` 中，如果 `SELECT` 一个常数，那么这个常数的值在查询结果中所有行都是一样的，比方说：

In [6]:
%%sql
SELECT
    EmployeeId,
    FirstName,
    LastName,
    1 AS ConstantValue
FROM Employee

,EmployeeId,FirstName,LastName,ConstantValue
0,1,Andrew,Adams,1
1,2,Nancy,Edwards,1
2,3,Jane,Peacock,1
3,4,Margaret,Park,1
4,5,Steve,Johnson,1
5,6,Michael,Mitchell,1
6,7,Robert,King,1
7,8,Laura,Callahan,1


但有的时候，我们并不想手写一个常数，而是从数据库中查出一个常数，这个时候就可以使用 `SELECT` 子句中的子查询来实现：

- 这个子查询结果必须是一个单值，也就是说，这个子查询必须只能返回一行一列的数据，否则会报错
- 如果子查询结果，是一个一列多行，或一行多列，或者多行多列的数据，那么在 `SELECT` 子句中的子查询就无法使用了，会报错
- 因为 SQL 并不知道要把这个多值的结果放在查询结果中的哪一行哪一列，所以就会报错

我们来看一个简单的例子：

In [15]:
%%sql
SELECT
    EmployeeId,
    FirstName,
    LastName,
    (SELECT COUNT(*) FROM Employee WHERE ReportsTo IS NOT NULL) AS EmployeeCount
FROM Employee

,EmployeeId,FirstName,LastName,EmployeeCount
0,1,Andrew,Adams,7
1,2,Nancy,Edwards,7
2,3,Jane,Peacock,7
3,4,Margaret,Park,7
4,5,Steve,Johnson,7
5,6,Michael,Mitchell,7
6,7,Robert,King,7
7,8,Laura,Callahan,7
